###Эксперимент 1. Анализ данных

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

In [ ]:
QWEN_CSV = "qwen.csv"
MISTRAL_CSV = "mistral.csv"

In [ ]:
import pandas as pd

human_df = pd.read_csv("human_reference.csv")
models_df = pd.read_csv("model_outputs.csv")

models = ["qwen", "mistral", "claude", "yandex", "alice"]

rows = []

for _, row in models_df.iterrows():
    base = {
        "id": row["id"],
        "verb": row["verb"],
        "meaning": row["meaning"]
    }

    for model in models:
        rows.append({
            **base,
            "model": model,
            "model_output": row.get(f"{model}_output", None),
            "model_suffix": row.get(f"{model}_suffix", None),
        })

model_long_df = pd.DataFrame(rows)


merged = model_long_df.merge(
    human_df,
    on=["id", "verb", "meaning"],
    how="left"
)


def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def parse_variants_field(x):
    if pd.isna(x) or str(x).strip() == "":
        return []

    variants = []
    groups = str(x).split(";")

    for group in groups:
        group = group.strip()
        if not group:
            continue

        subvariants = group.split("/")
        for v in subvariants:
            v = normalize_text(v)
            if v:
                variants.append(v)

    return variants

def parse_semicolon_list(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [normalize_text(item) for item in str(x).split(";") if normalize_text(item)]


merged["exact_majority_match"] = merged.apply(
    lambda r: int(
        normalize_text(r["model_output"]) in parse_variants_field(r["human_majority_form"])
    ),
    axis=1
)


merged["form_in_human_supported_set"] = merged.apply(
    lambda r: int(
        normalize_text(r["model_output"]) in parse_variants_field(r["human_supported_forms"])
    ),
    axis=1
)

merged["suffix_matches_majority"] = merged.apply(
    lambda r: int(
        normalize_text(r["model_suffix"]) == normalize_text(r["human_majority_suffix"])
    ),
    axis=1
)

merged["suffix_attested_by_humans"] = merged.apply(
    lambda r: int(
        normalize_text(r["model_suffix"]) in parse_semicolon_list(r["human_attested_suffixes"])
    ),
    axis=1
)

def classify_divergence(row):
    if row["exact_majority_match"] == 1:
        return "exact_majority_match"
    elif row["form_in_human_supported_set"] == 1:
        return "human_supported_non_majority_form"
    elif row["suffix_attested_by_humans"] == 1:
        return "new_form_attested_suffix"
    else:
        return "new_form_unattested_suffix"

merged["divergence_type"] = merged.apply(classify_divergence, axis=1)


merged.to_csv("model_vs_human_evaluation.csv", index=False, encoding="utf-8-sig")

print(merged[[
    "verb", "model", "model_output",
    "human_majority_form", "human_supported_forms",
    "exact_majority_match", "form_in_human_supported_set"
]].head(15))

In [ ]:
import pandas as pd

In [ ]:
eval_df = pd.read_csv("model_vs_human_evaluation.csv")

summary = eval_df.groupby("model").agg({
    "exact_majority_match": "mean",
    "form_in_human_supported_set": "mean",
    "suffix_matches_majority": "mean",
    "suffix_attested_by_humans": "mean"
}).reset_index()

summary = summary.rename(columns={
    "exact_majority_match": "exact_majority_match_rate",
    "form_in_human_supported_set": "human_supported_form_rate",
    "suffix_matches_majority": "majority_suffix_match_rate",
    "suffix_attested_by_humans": "human_attested_suffix_rate"
})

for col in [
    "exact_majority_match_rate",
    "human_supported_form_rate",
    "majority_suffix_match_rate",
    "human_attested_suffix_rate"
]:
    summary[col] = (summary[col] * 100).round(1)

summary.to_csv("model_summary.csv", index=False, encoding="utf-8-sig")
summary

In [ ]:
from collections import Counter

In [ ]:
human_df = pd.read_csv("human_reference.csv")
eval_df = pd.read_csv("model_vs_human_evaluation.csv")

In [ ]:
def parse_semicolon_list(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [item.strip() for item in str(x).split(";") if item.strip()]

In [ ]:
human_suffix_counts = (
    human_df["human_majority_suffix"]
    .value_counts(dropna=False)
    .reset_index()
)
human_suffix_counts.columns = ["suffix", "count"]
human_suffix_counts["source"] = "humans"
human_suffix_counts["share"] = (
    human_suffix_counts["count"] / human_suffix_counts["count"].sum() * 100
).round(1)

human_suffix_counts

In [ ]:
model_suffix_counts = (
    eval_df.groupby(["model", "model_suffix"])
    .size()
    .reset_index(name="count")
    .rename(columns={"model_suffix": "suffix"})
)

model_suffix_counts["share"] = (
    model_suffix_counts.groupby("model")["count"]
    .transform(lambda x: (x / x.sum() * 100).round(1))
)

model_suffix_counts.head()

In [ ]:
human_suffix_counts = human_suffix_counts[["source", "suffix", "count", "share"]].copy()
human_suffix_counts = human_suffix_counts.rename(columns={"source": "model"})

suffix_distribution_long = pd.concat(
    [
        human_suffix_counts,
        model_suffix_counts[["model", "suffix", "count", "share"]]
    ],
    ignore_index=True
)

suffix_distribution_long

In [ ]:
suffix_distribution_table = suffix_distribution_long.pivot(
    index="suffix",
    columns="model",
    values="share"
).fillna(0)

suffix_distribution_table

In [ ]:
suffix_distribution_counts = suffix_distribution_long.pivot(
    index="suffix",
    columns="model",
    values="count"
).fillna(0)

suffix_distribution_counts

In [ ]:
suffix_distribution_long.to_csv("suffix_distribution_long.csv", index=False, encoding="utf-8-sig")
suffix_distribution_table.to_csv("suffix_distribution_table.csv", encoding="utf-8-sig")
suffix_distribution_counts.to_csv("suffix_distribution_counts.csv", encoding="utf-8-sig")

In [ ]:
import pandas as pd

suffix_distribution_long = pd.read_csv("suffix_distribution_long.csv")
suffix_distribution_long.head()

In [ ]:
import matplotlib.pyplot as plt

plot_df = suffix_distribution_long.pivot(
    index="suffix",
    columns="model",
    values="share"
).fillna(0)

if "humans" in plot_df.columns:
    plot_df = plot_df.sort_values(by="humans", ascending=False)

gray_colors = ["#222222", "#4D4D4D", "#707070", "#909090", "#B0B0B0", "#C8C8C8"]

ax = plot_df.plot(
    kind="bar",
    figsize=(14, 7),
    width=0.85,
    color=gray_colors[:len(plot_df.columns)]
)


plt.title("Suffix distribution across humans and models")
plt.xlabel("Suffix")
plt.ylabel("Share (%)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Source")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

human_df = pd.read_csv("human_reference.csv")
eval_df = pd.read_csv("model_vs_human_evaluation.csv")

def parse_semicolon_list(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [item.strip() for item in str(x).split(";") if item.strip()]

human_all_suffixes = []

for _, row in human_df.iterrows():
    suffixes = parse_semicolon_list(row["human_attested_suffixes"])
    human_all_suffixes.extend(suffixes)

human_attested_suffix_counts = (
    pd.Series(human_all_suffixes)
    .value_counts(dropna=False)
    .reset_index()
)
human_attested_suffix_counts.columns = ["suffix", "count"]
human_attested_suffix_counts["model"] = "humans_attested"
human_attested_suffix_counts["share"] = (
    human_attested_suffix_counts["count"] / human_attested_suffix_counts["count"].sum() * 100
).round(1)

human_attested_suffix_counts